# Stellar Oblateness

## Purpose

This notebook calculates the stellar rotation rate, oblateness, and temperature distribution required for gravity-darkened transit modelling of HAT-P-70.

**Inputs**
- Stellar parameters from the literature
- Measured \(v \sin i\)

**Outputs**
- Stellar oblateness
- Rotation rate
- Polar and equatorial temperatures
- Parameters used by the STARRY model

# Step 1: Import necessary libraries

In [12]:
import numpy as np
import matplotlib.pyplot as plt

# Step 2 Provide example parameters

In [23]:
G = 6.67430e-11                 # Gravitational Constant
Vsini = 99_880                  # 99.9 km/s
R_star = 2.075 * 6.957e8        # R☉ in meters (Equatorial Radius)
M_star = 1.900 * 1.989e30       # M☉ in kg
T_eff = 8450                    # Kelvin 



# Step 3: Define the function

In [24]:
def inclination_constraints(Vsini, R_star, M_star, G=6.67430e-11):
    """
    Calculates f and omega for a grid of inclinations and identifies the range 
    of i_* where rotation remains sub-critical.
    
    Parameters:
    -----------
    Vsini   : float : observed V*sin(i) in m/s
    R_star  : float : stellar radius in meters
    M_star  : float : stellar mass in kg
    G       : float : gravitational constant (SI)
    
    Returns:
    --------
    valid_i  : ndarray : valid inclinations in degrees
    i_grid   : ndarray : full grid of inclinations
    f_vals   : ndarray : oblateness values
    omega    : ndarray : dimensionless angular velocities
    """
    
    # Avoid sin(0), use degrees from 1° to 179°
    i_grid = np.linspace(1, 179, 1000)
    sin_i = np.sin(np.radians(i_grid))
    
    # Equatorial velocity
    Veq = Vsini / sin_i  # m/s
    
    # Rotational period
    Prot = (2 * np.pi * R_star) / Veq  # seconds
    
    # Angular velocity
    Omega = 2 * np.pi / Prot
    
    # Dimensionless angular velocity
    omega = Omega * np.sqrt(R_star**3 / (G * M_star))
    
    # Oblateness from Roche model
    f_vals = 1 - (2 / (omega**2 + 2))
    
    # Valid inclinations: omega < 1 and f < 1/3
    valid_mask = (omega < 1) & (f_vals < 1/3)
    valid_i = i_grid[valid_mask]
    
    return valid_i, i_grid, f_vals, omega

# Step 4: Call function 

In [25]:
valid_i, i_grid, f_vals, omega_vals = inclination_constraints(Vsini, R_star, M_star)


# Step 5: Display the result

In [26]:
if len(valid_i) > 0:
    print(f"✅ Valid inclination range (i*) before critical rotation: {valid_i[0]:.2f}° to {valid_i[-1]:.2f}°")
else:
    print("⚠️ No valid inclination range found — star may already be rotating critically.")


✅ Valid inclination range (i*) before critical rotation: 13.83° to 166.17°


# Step 6 - Compute key values at i_* = 90° (lowest limit)

In [27]:
# Compute key values at i_* = 90° 

# Equator-on case: i_* = 90°
Veq = Vsini                     # since sin(90°) = 1

# Rotational period
Prot_eq = (2 * np.pi * R_star) / Veq  # in seconds
Prot_days = Prot_eq / 86400           # convert to days

# Angular velocity
Omega_eq = 2 * np.pi / Prot_eq  # rad/s

# Dimensionless rotation rate
omega_eq = Omega_eq * np.sqrt(R_star**3 / (G * M_star))

# Oblateness from Roche approximation
f_eq = 1 - (2 / (omega_eq**2 + 2))

# Print results
print("\n✅System Rotation Values (i* = 90°):")
print(f" - Rotational period (Prot): {Prot_days:.3f} days")
print(f" - Angular velocity (Omega): {Omega_eq:.3e} rad/s")
print(f" - Dimensionless spin rate (omega): {omega_eq:.3f}")
print(f" - Oblateness (f): {f_eq:.4f}")



✅System Rotation Values (i* = 90°):
 - Rotational period (Prot): 1.051 days
 - Angular velocity (Omega): 6.919e-05 rad/s
 - Dimensionless spin rate (omega): 0.239
 - Oblateness (f): 0.0278


# Step 7 - Investigate f with different values of i*

In [28]:
# Set a specific i_* value (in degrees)
i_input = 38.93 # or any inclination we want to try

# Re-run constraint function
_, i_grid, f_vals, omega_vals = inclination_constraints(Vsini, R_star, M_star)

# Find closest index
idx = (np.abs(i_grid - i_input)).argmin()

# Extract corresponding f and omega
f_for_input_i = f_vals[idx]
omega_for_input_i = omega_vals[idx]

# Print results
print(f"i_grid = {i_grid [idx]}")
print(f"For i* = {i_input}°:")
print(f" - Oblateness f ≈ {f_for_input_i:.4f}")
print(f" - Dimensionless angular velocity ω ≈ {omega_for_input_i:.4f}")

# Check if valid
if f_for_input_i < 1/3:
    print("✅ This value is within the allowed oblateness limit (f < 1/3).")
else:
    print("⚠️ Warning: This value exceeds the oblateness limit (f ≥ 1/3).")


i_grid = 38.951951951951955
For i* = 38.93°:
 - Oblateness f ≈ 0.0674
 - Dimensionless angular velocity ω ≈ 0.3801
✅ This value is within the allowed oblateness limit (f < 1/3).


# Step 8 Define a new function which generate Tpole (Dholakia) and Tequator (von Zeipel)

In [34]:
def estimate_T_equator_from_Teff(T_eff, omega, beta=0.25):
    """
    Estimate the equatorial temperature from the effective temperature
    using a simple gravity-darkening correction based on oblateness.
    """
    f = 1 - 2 / (omega**2 + 2)  # oblateness factor

    # As the star spins faster (larger f), the equator cools down
    #T_eq = T_eff * (1 - f)**(beta/4) divide by four if is to be applied to flux
    T_eq = T_eff * (1 - f)**(beta) # do not divide by four if is to be applied to temperatures
    return T_eq


def calculate_T_pole(T_equator, omega, R_star, M_star, beta=0.25, G=6.67430e-11):
    """
    Calculate the polar temperature of a gravity-darkened star.
    """
    f = 1 - 2 / (omega**2 + 2)
    R_pole = (1 - f)*R_star

    # Surface gravities at equator and pole
    g_eq = G * M_star / (R_star**2)
    g_pole = G * M_star / (R_star**2 * (1 - f)**2)

    # Apply von Zeipel's law
    T_pole = T_equator * (g_pole / g_eq)**beta
    return T_pole, R_pole, g_pole, g_eq


# --- Use Teff from Step 2 ---
star_f = f_for_input_i
#beta = 0.2
beta = (
    -74.61562401*star_f**5
    +53.25102994*star_f**4
    -14.26673657*star_f**3
    +1.73631257*star_f**2
    -0.40595372*star_f
    +0.25070391
    )                    # Espinosa Lara
omega = omega_for_input_i 

# T_eff, R_star, M_star, and i_input should already be defined earlier
T_equator = estimate_T_equator_from_Teff(T_eff, omega, beta=beta)
T_pole, R_p, g_p, g_e = calculate_T_pole(T_equator, omega, R_star, M_star, beta=beta)

print(f"For T_eff = {T_eff} K, i* = {i_input} deg, β = {beta:.4f} and ω = {omega:.4f}, "
      f"T_equator ≈ {T_equator:.2f} K, T_pole ≈ {T_pole:.2f} K, f = {star_f:.4f}")
print(R_p/6.957e8, np.log10(g_p)+2)

For T_eff = 8450 K, i* = 38.93 deg, β = 0.2279 and ω = 0.3801, T_equator ≈ 8316.77 K, T_pole ≈ 8585.36 K, f = 0.0674
1.935215487178385 4.143491439124398
